# Notebook 1: Food Recognition Foundations with Fruits-360
### Minor: AI for Society | Personal Project: Food Recognition & Calorie Estimator
**Author:** Storm Burg  
**Dataset:** Fruits-360  
**Objective:** Learn computer vision fundamentals by training a Convolutional Neural Network (CNN) **from scratch** to classify a subset of raw fruits and vegetables.

---
## Theoretical Context & Project Roadmap
1. **Why start with Fruits-360 before complex dishes (like Food-101)?**
   - In computer vision, isolated objects on clean backgrounds allow us to verify that a model learns discriminative morphological and color features rather than memorizing background noise (tables, plates, cutlery).
   - This baseline gives us an intuitive, interpretable proof of concept before tackling complex multi-ingredient meal datasets and nutrition regressions.
2. **Learning Outcomes:**
   - Understand how digital images are represented as tensors.
   - Construct convolutional, pooling, batch normalization, and fully-connected layers from mathematical intuition.
   - Design data augmentations without corrupting semantic identities.
   - Track training dynamics (loss surfaces, accuracy, overfitting) and evaluate errors using confusion matrices.


## 1. Setup & Environment
We verify GPU availability (an NVIDIA T4 or similar in Google Colab), set random seeds for reproducibility, and install/import required libraries.


In [ ]:
# Install helper libraries if not present
!pip install -q kagglehub torch torchvision matplotlib seaborn scikit-learn tqdm

import os
import random
import shutil
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

# Set reproducible seeds
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"Device name: {torch.cuda.get_device_name(0)}")


## 2. Mount Google Drive for Checkpoint Persistence
Colab instances are ephemeral. If your session disconnects, any unsaved files will be lost.  
Here, we mount Google Drive and configure the directory (`Minor AI/models/`) to persist our model weights and training artifacts.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Directory on Google Drive for checkpoints
SAVE_DIR = '/content/drive/MyDrive/Minor AI/models'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Checkpoint directory ready: {SAVE_DIR}")


## 3. Dataset Acquisition & Subset Selection
Fruits-360 contains over 100 classes with specific varieties (e.g. 10 types of apples).  
To keep our initial CNN experiment clean, interpretable, and balanced, we filter the dataset to a distinct 12-class subset of common fruits and vegetables:
- **Apple (Braeburn)**
- **Banana**
- **Lemon**
- **Orange**
- **Strawberry**
- **Tomato**
- **Corn**
- **Carrot**
- **Cucumber (Ripe)**
- **Pepper Bell**
- **Avocado**
- **Pineapple**


In [ ]:
import kagglehub

# Download Fruits-360 via kagglehub
print("Downloading / verifying Fruits-360 dataset...")
raw_path = kagglehub.dataset_download("moltean/fruits")
print(f"Dataset downloaded to: {raw_path}")

# Locate Training and Test directories
train_source = os.path.join(raw_path, "fruits-360_dataset", "fruits-360", "Training")
test_source = os.path.join(raw_path, "fruits-360_dataset", "fruits-360", "Test")

# Fallback check if folder structure varies slightly
if not os.path.exists(train_source):
    train_source = os.path.join(raw_path, "Training")
    test_source = os.path.join(raw_path, "Test")

print(f"Training source directory: {train_source}")
print(f"Test source directory: {test_source}")


### Create a Focused Local Subset
We copy only our 12 chosen categories to local ephemeral Colab storage (`/content/dataset_subset/`). This ensures fast I/O speeds and clean data handling.


In [ ]:
SELECTED_CLASSES = [
    "Apple Braeburn",
    "Banana",
    "Lemon",
    "Orange",
    "Strawberry",
    "Tomato 1",
    "Corn",
    "Carrot",
    "Cucumber Ripe",
    "Pepper Bell",
    "Avocado",
    "Pineapple"
]

LOCAL_DATA_DIR = "/content/dataset_subset"
local_train = os.path.join(LOCAL_DATA_DIR, "train")
local_test = os.path.join(LOCAL_DATA_DIR, "test")

# Recreate folders
if os.path.exists(LOCAL_DATA_DIR):
    shutil.rmtree(LOCAL_DATA_DIR)
os.makedirs(local_train, exist_ok=True)
os.makedirs(local_test, exist_ok=True)

# Helper function to find best matching directory name
available_train_folders = os.listdir(train_source)

def copy_class_data(class_target, src_root, dst_root):
    # Match exact or case-insensitive prefix
    matched = None
    for folder in os.listdir(src_root):
        if folder.lower() == class_target.lower():
            matched = folder
            break
    if not matched:
        # fuzzy match
        for folder in os.listdir(src_root):
            if class_target.lower() in folder.lower():
                matched = folder
                break
    if matched:
        src_folder = os.path.join(src_root, matched)
        dst_folder = os.path.join(dst_root, class_target.replace(" ", "_"))
        shutil.copytree(src_folder, dst_folder)
        return True
    return False

print("Copying selected classes...")
for cls in SELECTED_CLASSES:
    copied_tr = copy_class_data(cls, train_source, local_train)
    copied_te = copy_class_data(cls, test_source, local_test)
    print(f"  {cls}: {'[OK]' if (copied_tr and copied_te) else '[MISSING]'}")

active_classes = sorted(os.listdir(local_train))
print(f"\nReady with {len(active_classes)} categories: {active_classes}")


## 4. Exploratory Data Analysis (EDA)
Before training any machine learning model, we must inspect our inputs:
1. **Class Distribution**: Are our training and test sets balanced?
2. **Visual Inspection**: What do the raw pixels and spatial compositions look like?
3. **Color Channels & Shapes**: Validate dimensions (100x100x3).


In [ ]:
# 1. Inspect sample counts per category
train_counts = {cls: len(os.listdir(os.path.join(local_train, cls))) for cls in active_classes}
test_counts = {cls: len(os.listdir(os.path.join(local_test, cls))) for cls in active_classes}

plt.figure(figsize=(12, 4))
x = np.arange(len(active_classes))
width = 0.35

plt.bar(x - width/2, [train_counts[c] for c in active_classes], width, label='Train', color='#2563eb')
plt.bar(x + width/2, [test_counts[c] for c in active_classes], width, label='Test', color='#10b981')
plt.xticks(x, active_classes, rotation=45, ha='right')
plt.title("Sample Count per Class in Selected Subset")
plt.ylabel("Number of Images")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Total training samples: {sum(train_counts.values())}")
print(f"Total test samples: {sum(test_counts.values())}")


In [ ]:
# 2. Visualize Sample Images across All Selected Classes
plt.figure(figsize=(14, 6))
cols = 6
rows = int(np.ceil(len(active_classes) / cols))

for idx, cls in enumerate(active_classes):
    cls_folder = os.path.join(local_train, cls)
    sample_file = random.choice(os.listdir(cls_folder))
    img = Image.open(os.path.join(cls_folder, sample_file))
    
    plt.subplot(rows, cols, idx + 1)
    plt.imshow(img)
    plt.title(cls, fontsize=10)
    plt.axis("off")

plt.suptitle("Sample Images from Fruits-360 Subset", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


## 5. Data Preprocessing & Augmentations
### The Math & Rationale
1. **Random Horizontal Flip ($p=0.5$):** Reflecting a fruit across its vertical axis does not alter what fruit it is, effectively doubling orientation diversity.
2. **Random Rotation ($\pm 15^\circ$):** Accounts for camera angle variations when photographing real foods.
3. **Carefully Controlled Color Jitter:** Small adjustments to brightness and contrast simulate varied indoor lighting. **Crucial restriction:** Hue is kept unchanged or barely tweaked ($\le 0.02$) because color is an essential semantic feature distinguishing a lemon from a lime or an orange from a tomato.
4. **Normalization:** Transforming pixel values from $[0, 255]$ to standardized distributions with mean $\mu$ and standard deviation $\sigma$:
   $$x_{norm} = \frac{x - \mu}{\sigma}$$
   This keeps activation distributions well-behaved across gradient backpropagation steps.


In [ ]:
# Define transforms
train_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

test_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Load datasets via torchvision ImageFolder
train_dataset = datasets.ImageFolder(root=local_train, transform=train_transforms)
test_dataset = datasets.ImageFolder(root=local_test, transform=test_transforms)

# DataLoaders
BATCH_SIZE = 64
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"DataLoaders ready:")
print(f"  Train batches: {len(train_loader)} (Batch size: {BATCH_SIZE})")
print(f"  Test batches:  {len(test_loader)} (Batch size: {BATCH_SIZE})")


## 6. Building the CNN Architecture From Scratch

### Convolutional Neural Network Anatomy
A standard vision model processes hierarchical features:
1. **Early layers (Conv1):** Learn low-level primitives: edges, gradients, color contrasts.
2. **Intermediate layers (Conv2 & Conv3):** Combine primitives into textures and simple shapes (curves, fruit skins, stems).
3. **Deep layers (Dense/Linear):** High-level semantic reasoning (e.g. combining yellow color + elongated curved shape = banana).

```
Input Image (3 x 100 x 100)
    │
    ▼
[Conv2d (3 -> 32, 3x3) + BatchNorm2d + ReLU] ──> MaxPool2d (2x2) ──> (32 x 50 x 50)
    │
    ▼
[Conv2d (32 -> 64, 3x3) + BatchNorm2d + ReLU] ──> MaxPool2d (2x2) ──> (64 x 25 x 25)
    │
    ▼
[Conv2d (64 -> 128, 3x3) + BatchNorm2d + ReLU] ──> MaxPool2d (2x2) ──> (128 x 12 x 12)
    │
    ▼
[AdaptiveAvgPool2d (4x4)] ───────────────────────────────────────────> (128 x 4 x 4 = 2048)
    │
    ▼
[Flatten] ──> [Linear (2048 -> 256) + ReLU + Dropout(0.4)]
    │
    ▼
[Linear (256 -> num_classes)] ───────────────────────────────────────> Raw Logits
```

### Why Batch Normalization & Dropout?
- **Batch Normalization:** Normalizes intermediate layer activations, accelerating convergence and allowing higher learning rates.
- **Dropout ($p=0.4$):** Randomly zeroes out neurons during training, preventing individual units from co-adapting and reducing overfitting.


In [ ]:
class FruitClassifierCNN(nn.Module):
    def __init__(self, num_classes=len(active_classes)):
        super(FruitClassifierCNN, self).__init__()
        
        # Block 1: Low-level features
        self.block1 = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)  # 100x100 -> 50x50
        )
        
        # Block 2: Textures & mid-level patterns
        self.block2 = nn.Sequential(
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)  # 50x50 -> 25x25
        )
        
        # Block 3: Complex shapes & contours
        self.block3 = nn.Sequential(
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)  # 25x25 -> 12x12
        )
        
        # Spatial reduction
        self.pool = nn.AdaptiveAvgPool2d((4, 4)) # 128 x 4 x 4 = 2048 features
        
        # Classification Head
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(p=0.4),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.pool(x)
        logits = self.classifier(x)
        return logits

model = FruitClassifierCNN(num_classes=len(active_classes)).to(device)

# Count trainable parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"\nTotal Trainable Parameters: {total_params:,}")


## 7. Model Training & Optimization Loop
- **Loss Function:** `nn.CrossEntropyLoss()` combines LogSoftmax and Negative Log-Likelihood Loss.
- **Optimizer:** `Adam(lr=0.001)` with adaptive momentum per-parameter.
- **Scheduler:** `ReduceLROnPlateau` lowers learning rate if test loss ceases to improve.
- **Checkpointing:** Saves best weights based on validation accuracy directly to Google Drive.


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2, verbose=True)

EPOCHS = 15
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': []
}

best_val_acc = 0.0
checkpoint_path = os.path.join(SAVE_DIR, 'best_fruits_cnn.pt')

print(f"Starting training for {EPOCHS} epochs on {device}...")

for epoch in range(1, EPOCHS + 1):
    # ---- TRAINING PHASE ----
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [Train]"):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct_train += (preds == labels).sum().item()
        total_train += labels.size(0)
        
    epoch_train_loss = running_loss / total_train
    epoch_train_acc = (correct_train / total_train) * 100.0
    
    # ---- VALIDATION / TEST PHASE ----
    model.eval()
    val_loss = 0.0
    correct_val = 0
    total_val = 0
    
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct_val += (preds == labels).sum().item()
            total_val += labels.size(0)
            
    epoch_val_loss = val_loss / total_val
    epoch_val_acc = (correct_val / total_val) * 100.0
    
    # Adjust learning rate based on validation accuracy
    scheduler.step(epoch_val_acc)
    
    # Track stats
    history['train_loss'].append(epoch_train_loss)
    history['train_acc'].append(epoch_train_acc)
    history['val_loss'].append(epoch_val_loss)
    history['val_acc'].append(epoch_val_acc)
    
    print(f"Epoch [{epoch:02d}/{EPOCHS}] | "
          f"Train Loss: {epoch_train_loss:.4f} - Train Acc: {epoch_train_acc:.2f}% | "
          f"Val Loss: {epoch_val_loss:.4f} - Val Acc: {epoch_val_acc:.2f}%")
    
    # Save best model
    if epoch_val_acc > best_val_acc:
        best_val_acc = epoch_val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_val_acc': best_val_acc,
            'classes': active_classes
        }, checkpoint_path)
        print(f"  --> Saved new best checkpoint to {checkpoint_path} ({best_val_acc:.2f}%)")

print(f"\nTraining complete! Best Validation Accuracy: {best_val_acc:.2f}%")


## 8. Quantitative Evaluation & Learning Curves
We plot the training vs. validation loss and accuracy curves to diagnose training dynamics:
- **Underfitting:** High training loss and high validation loss.
- **Overfitting:** Training loss continues falling while validation loss climbs.
- **Good convergence:** Both loss curves decrease smoothly and plateau.


In [ ]:
plt.figure(figsize=(14, 5))

# Loss plot
plt.subplot(1, 2, 1)
plt.plot(range(1, EPOCHS + 1), history['train_loss'], label='Train Loss', marker='o')
plt.plot(range(1, EPOCHS + 1), history['val_loss'], label='Val Loss', marker='s')
plt.title("Loss Curves")
plt.xlabel("Epoch")
plt.ylabel("Cross-Entropy Loss")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

# Accuracy plot
plt.subplot(1, 2, 2)
plt.plot(range(1, EPOCHS + 1), history['train_acc'], label='Train Acc', marker='o')
plt.plot(range(1, EPOCHS + 1), history['val_acc'], label='Val Acc', marker='s')
plt.title("Accuracy Curves")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()


## 9. Error Analysis: Confusion Matrix & Per-Class Accuracy
Overall accuracy can obscure specific misclassifications (e.g. confusing an orange with a tomato).  
A confusion matrix visualizes how predictions map against ground truth labels.


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

# Load best weights for evaluation
checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

all_preds = []
all_targets = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(labels.numpy())

# Confusion Matrix
cm = confusion_matrix(all_targets, all_preds)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=active_classes,
            yticklabels=active_classes)
plt.title("Confusion Matrix (Best Model Checkpoint)")
plt.xlabel("Predicted Class")
plt.ylabel("True Class")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Classification Report
print("\nClassification Report:")
print(classification_report(all_targets, all_preds, target_names=active_classes, digits=3))


## 10. Qualitative Prediction Inspection
Let's inspect individual model predictions on test images, displaying both confidence levels and color-coded outcomes (Green = Correct, Red = Misclassified).


In [ ]:
# Helper to denormalize for visualization
def denormalize(tensor):
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = tensor.cpu().numpy().transpose((1, 2, 0))
    img = std * img + mean
    return np.clip(img, 0, 1)

# Pick random samples from test set
model.eval()
indices = random.sample(range(len(test_dataset)), 12)

plt.figure(figsize=(15, 8))
cols = 4
rows = 3

with torch.no_grad():
    for plot_idx, idx in enumerate(indices):
        img_tensor, true_label = test_dataset[idx]
        output = model(img_tensor.unsqueeze(0).to(device))
        probs = torch.softmax(output, dim=1)
        conf, pred_label = torch.max(probs, 1)
        
        pred_cls = active_classes[pred_label.item()]
        true_cls = active_classes[true_label]
        is_correct = (pred_label.item() == true_label)
        
        plt.subplot(rows, cols, plot_idx + 1)
        plt.imshow(denormalize(img_tensor))
        color = 'green' if is_correct else 'red'
        plt.title(f"True: {true_cls}\nPred: {pred_cls} ({conf.item()*100:.1f}%)", color=color, fontsize=10)
        plt.axis("off")

plt.suptitle("Sample Predictions on Test Set", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


## 11. Reflection, Key Learnings & Next Steps

### What We Learned in This Experiment:
1. **CNN Mechanics:** Convolutions successfully extracted spatial and color-based features without manual feature engineering.
2. **From-Scratch Feasibility:** On a clean dataset with clear visual contrasts, a 3-layer CNN achieved high accuracy (>90%) within 15 epochs.
3. **Data Quality Impact:** Uniform lighting and segmented backgrounds make learning significantly easier compared to complex, multi-ingredient dish photography.

---
### Bridging to Notebook 2: Scaling to Complex Meals (Food-101 & Transfer Learning)
In the next stage of our project:
- We will transition to **Food-101**, containing real meal photos with complex backgrounds, mixed textures, sauces, and garnishes.
- Because training a deep model from scratch on 101,000 real-world dishes requires weeks of compute and struggles with noisy labels, we will implement **Transfer Learning** using a pretrained backbone (e.g. EfficientNet or ResNet) fine-tuned for dish classification.
- In **Notebook 3**, we will link these dish classifications to a **calorie regression model** and portion-size multipliers.
